<a href="https://colab.research.google.com/github/tarun161/AILearning/blob/ITSupportAgent/ITSupportAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install langchain langchain-openai langchain-core langchain_classic

In [15]:
import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain_core.tools import tool
from google.colab import userdata
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent


In [17]:
@tool
def get_server_health(server_id: str) -> str:
    """Returns CPU and Memory usage for a given server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    result = metrics.get(server_id, {"error": "Server not found. Check the ID."})
    return json.dumps(result)

@tool
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Returns the last N lines of logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    # Different logs for different servers to trigger different agent behaviors
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

@tool
def restart_service(server_id: str) -> str:
    """Restarts Server"""
    print(f'Restarting service {server_id}')
    response = f'{{"status": "success", "message": "Server restarted successfully", "server_id":"{server_id}"}}'
    return json.dumps(response)

@tool
def escalate_to_engineer(summary: str) -> str:
    """Escalates summary to Engineer"""
    print('-> TOOL: Escalating to human...')
    response = f'{{"status": "success", "message": "Ticket created", "summary":"{summary}"}}'
    return json.dumps(response)

In [22]:
tools = [get_server_health,fetch_recent_logs,restart_service,escalate_to_engineer]
openAi = ChatOpenAI(model="gpt-4o", temperature=0.7, api_key=userdata.get("OPENAI_KEY"))
prompt = ChatPromptTemplate.from_messages([
   ( "system", "You are a Level 1 IT Responder. Investigate server issues. If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer."),
   ( "user", "{user_issue}"),
   ("placeholder", "{agent_scratchpad}")
])
agent = create_tool_calling_agent(openAi,tools,prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
def run_it_agent(user_issue : str) :
  response = agent_executor.invoke({"user_issue": user_issue})
  return response['output']





In [23]:
# Scenario A: Should trigger a restart (CPU is 98%)
run_it_agent("The payment-server-01 is extremely slow and timing out.")



> Entering new AgentExecutor chain...

Invoking: `get_server_health` with `{'server_id': 'payment-server-01'}`


-> TOOL: Checking health for payment-server-01...
{"cpu": "98%", "memory": "40%", "status": "Warning"}
Invoking: `fetch_recent_logs` with `{'server_id': 'payment-server-01', 'lines': 10}`


-> TOOL: Fetching last 10 log lines for payment-server-01...
{"logs": ["[INFO] Request received /pay/v1", "[WARN] CPU threshold exceeded 90%", "[WARN] Thread pool exhaustion", "[CRITICAL] Process hung, not accepting new connections", "[ERROR] Timeout waiting for thread"]}
Invoking: `restart_service` with `{'server_id': 'payment-server-01'}`


Restarting service payment-server-01
"{\"status\": \"success\", \"message\": \"Server restarted successfully\", \"server_id\":\"payment-server-01\"}"The server `payment-server-01` had a CPU usage of 98%, which was causing the slowness and timeouts. I have successfully restarted the service to address this issue. Please monitor the server and let me

'The server `payment-server-01` had a CPU usage of 98%, which was causing the slowness and timeouts. I have successfully restarted the service to address this issue. Please monitor the server and let me know if you need further assistance.'

In [24]:
# Scenario B: Should trigger an escalation (DB is healthy but logs might be weird)
run_it_agent("Something is wrong with db-node-02")



> Entering new AgentExecutor chain...

Invoking: `get_server_health` with `{'server_id': 'db-node-02'}`


-> TOOL: Checking health for db-node-02...
{"cpu": "12%", "memory": "60%", "status": "Healthy"}
Invoking: `fetch_recent_logs` with `{'server_id': 'db-node-02', 'lines': 5}`


-> TOOL: Fetching last 5 log lines for db-node-02...
{"logs": ["[INFO] Backup started", "[INFO] Backup completed successfully", "[INFO] User query executed in 12ms", "[INFO] Health check: OK", "[INFO] Replication sync active"]}The server `db-node-02` appears to be healthy in terms of CPU and memory usage, with CPU at 12% and memory at 60%. The recent logs also do not indicate any critical issues; they show normal operations such as backups, user queries, health checks, and replication sync activities.

It seems there might not be an issue with the server at the moment. If you have observed specific symptoms or issues, please provide more details so I can assist you further.

> Finished chain.


'The server `db-node-02` appears to be healthy in terms of CPU and memory usage, with CPU at 12% and memory at 60%. The recent logs also do not indicate any critical issues; they show normal operations such as backups, user queries, health checks, and replication sync activities.\n\nIt seems there might not be an issue with the server at the moment. If you have observed specific symptoms or issues, please provide more details so I can assist you further.'

In [25]:
# Scenario C: The High Memory Case (auth-service-03)
# Agent should see Memory 95% + OutOfMemoryError logs -> Restart
run_it_agent("Users are reporting login failures on auth-service-03.")

print("\n" + "="*50 + "\n")



> Entering new AgentExecutor chain...

Invoking: `get_server_health` with `{'server_id': 'auth-service-03'}`


-> TOOL: Checking health for auth-service-03...
{"cpu": "45%", "memory": "95%", "status": "Critical"}
Invoking: `fetch_recent_logs` with `{'server_id': 'auth-service-03', 'lines': 5}`


-> TOOL: Fetching last 5 log lines for auth-service-03...
{"logs": ["[INFO] Token validated user_882", "[WARN] Garbage collection taking too long (>5s)", "[ERROR] java.lang.OutOfMemoryError: Java heap space", "[CRITICAL] Application crashing due to memory leak", "[INFO] Restarting context..."]}
Invoking: `restart_service` with `{'server_id': 'auth-service-03'}`


Restarting service auth-service-03
"{\"status\": \"success\", \"message\": \"Server restarted successfully\", \"server_id\":\"auth-service-03\"}"The auth-service-03 server was experiencing high memory usage, which was causing application crashes due to a memory leak. I have successfully restarted the service to address this issue. Pl

In [26]:
# Scenario D: The Dependency Failure (search-index-09)
# Agent should see healthy CPU but "Connection Refused" logs -> Escalate
run_it_agent("Search isn't working. Can you check search-index-09?")

print("\n" + "="*50 + "\n")



> Entering new AgentExecutor chain...

Invoking: `get_server_health` with `{'server_id': 'search-index-09'}`


-> TOOL: Checking health for search-index-09...
{"cpu": "10%", "memory": "15%", "status": "Error"}
Invoking: `fetch_recent_logs` with `{'server_id': 'search-index-09', 'lines': 5}`


-> TOOL: Fetching last 5 log lines for search-index-09...
{"logs": ["[INFO] Indexing started", "[ERROR] Connection refused: elastic-cluster-main:9200", "[ERROR] Failed to write document ID 4432", "[CRITICAL] Dependency Unreachable: Search Engine is down", "[ERROR] Retrying in 30s..."]}
Invoking: `escalate_to_engineer` with `{'summary': "The search-index-09 server is experiencing critical dependency issues. Logs show connection refused errors to the elastic-cluster-main:9200 and a critical error stating 'Dependency Unreachable: Search Engine is down'. CPU and Memory usage are normal, but the service is unable to connect to its main search engine dependency, which a restart won't fix."}`


-> TOOL

In [27]:
# Scenario E: The Healthy Server (frontend-node-04)
# Agent should see normal stats and 200 OK logs -> Do nothing / Report healthy
run_it_agent("Check frontend-node-04 just to be safe.")



> Entering new AgentExecutor chain...

Invoking: `get_server_health` with `{'server_id': 'frontend-node-04'}`


-> TOOL: Checking health for frontend-node-04...
{"cpu": "25%", "memory": "30%", "status": "Healthy"}The server "frontend-node-04" is currently healthy with CPU usage at 25% and memory usage at 30%. There are no immediate issues that require attention. If there's anything else you'd like me to check or assist with, please let me know!

> Finished chain.


'The server "frontend-node-04" is currently healthy with CPU usage at 25% and memory usage at 30%. There are no immediate issues that require attention. If there\'s anything else you\'d like me to check or assist with, please let me know!'